# Group 2 - Redis

**Capital Bikeshare** provides a reliable source of transportation for residents and visitors of Washington DC. Operations analysts would benefit greatly from insights about how DC residents use the system to travel. Information about the *most used stations, most common routes, and other bike usage patterns* would help Capital Bikeshare figure out where to allocate resources. 

**Redis** can be used as a strong database because a lot of the useful information that will be accessed frequently can be represented in data structures that are consistently in the same format. For example, a sorted set can be used to display rankings for the top 10 stations or travel routes in the data. Rather than querying and aggregating individual trip records every time we want to see the ranking, Redis allows us to maintain an ongoing ranked list that can be accessed directly.

We are using [Trip History data](https://capitalbikeshare.com/system-data) from September 2026. The data contains the following features:

* `ride_id` - Unique ID for the ride

* `rideable_type` - Whether the bike rode was a classic or electric bike

* `started_at` - Start date and time

* `ended_at` - End date and time

* `start_station_name` and `start_station_id` - Name and identifier of the starting station. 
    * No value if the ride was started with the bike undocked. This is only possible with electric bikes.

* `end_station_name` and `end_station_id` - Name and identifier of the ending station.
    * No value if the ride was ended with the bike undocked. This is only possible with electric bikes.

* `start_lat`, `start_lng`, `end_lat`, `end_lng` - Start and end coordinates
    * If the bike is docked at a station, the coordinates of that station (precision of 6) are used regardless of bike type. 
    * If the bike is undocked (precision of 2), which is only possible with electric bikes, the bike's direct coordinates are used.

* `member_casual` - Indicates whether user was a registered "member" (Annual Member, 30-Day Member or Day Key Member) or a "casual" rider (Single Trip, 24-Hour Pass, 3-Day Pass or 5-Day Pass)
    * For `casual` riders, bike unlocks are $1. Classic bikes are $0.15/min. Electric bikes are $0.35/min.
        * Parking an electric bike outside of a capital bikeshare station costs $3.
        * Casual riders who purchase a Day Pass ($10/day) fall under the same pricing scheme as member riders. As there is no distinction made between these plans in the dataset, we will not consider Day Pass pricing when estimating the trip cost.

    * For `member` riders, bike unlocks are free. Classic bikes are free for 45 minutes, then $0.05/min. Electric bikes are $0.15/min.
        * Parking an electric bike outside of a capital bikeshare station costs $3. 
        * If the rider parks next to a full station with 3 or fewer docks available, there is no additional charge. Due to lack of precision in undocked electric bike coordinates, we will not consider this case.

This data has been processed before release to remove trips taken by staff as they service and inspect the system, trips that are taken to/from any of “test” stations at Capital Bikeshare warehouses, and any trips lasting less than 60 seconds.

In [3]:
import pandas as pd

In [43]:
pd.__version__

'3.1.0'

In [4]:
# read in the Capital Bikeshare data
df_cabi = pd.read_csv("202609-capitalbikeshare-tripdata/202609-capitalbikeshare-tripdata.csv")

# cast start and end times to datetime
df_cabi["started_at"] = pd.to_datetime(df_cabi["started_at"])
df_cabi["ended_at"] = pd.to_datetime(df_cabi["ended_at"])

# cast station IDs to string
df_cabi["start_station_id"] = df_cabi["start_station_id"].astype('Int64').astype(str)
df_cabi["end_station_id"] = df_cabi["end_station_id"].astype('Int64').astype(str)

In [34]:
# head of the dataframe
df_cabi.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,3FA776BA68AAC5F2,classic_bike,2026-09-13 13:29:42.674,2026-09-13 13:40:33.637,17th St & Potomac Ave SE / Congressional Cemet...,31727,17th St & Potomac Ave SE / Congressional Cemet...,31727,38.882458,-76.980715,38.882458,-76.980715,member
1,0DDE689949EEE2AA,classic_bike,2026-09-13 16:02:48.322,2026-09-13 16:16:16.344,17th St & Potomac Ave SE / Congressional Cemet...,31727,17th St & Potomac Ave SE / Congressional Cemet...,31727,38.882458,-76.980715,38.882458,-76.980715,member
2,4B620586F5D84EC8,electric_bike,2026-09-12 11:02:15.677,2026-09-12 13:26:22.000,Fenton St & New York Ave,32001,Fenton St & New York Ave,32001,38.985404,-77.023082,38.985404,-77.023082,casual
3,CB1C2CADBB6C64AD,electric_bike,2026-09-01 20:49:14.502,2026-09-01 21:05:21.948,American University East Campus,31320,American University East Campus,31320,38.936298,-77.087128,38.936298,-77.087128,member
4,E3F0AA83CFB29B45,electric_bike,2026-09-18 23:17:01.401,2026-09-18 23:22:47.884,MLK & Marion Barry Ave SE,31802,16th & Bangor St SE,31829,38.867373,-76.988039,38.860440,-76.983585,member


In [5]:
df_cabi.shape

(676808, 13)

In [7]:
# column information
df_cabi.info()

<class 'pandas.DataFrame'>
RangeIndex: 676808 entries, 0 to 676807
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   ride_id             676808 non-null  str           
 1   rideable_type       676808 non-null  str           
 2   started_at          676808 non-null  datetime64[us]
 3   ended_at            676808 non-null  datetime64[us]
 4   start_station_name  576363 non-null  str           
 5   start_station_id    576363 non-null  str           
 6   end_station_name    571291 non-null  str           
 7   end_station_id      571107 non-null  str           
 8   start_lat           676808 non-null  float64       
 9   start_lng           676808 non-null  float64       
 10  end_lat             676392 non-null  float64       
 11  end_lng             676392 non-null  float64       
 12  member_casual       676808 non-null  str           
dtypes: datetime64[us](2), float64(4), str(7)

In [8]:
# unique start and end stations
print(f"There are {df_cabi["start_station_name"].nunique()} unique start stations and " \
      f"{df_cabi["end_station_name"].nunique()} unique end stations.")

There are 865 unique start stations and 866 unique end stations.


In [9]:
# number of rides
len(df_cabi)

676808

In [10]:
# percent of rides with no start station, no end station, or neither
null_start_only = df_cabi[df_cabi["start_station_name"].isnull() & df_cabi["end_station_name"].notnull()].shape[0]
print(f"{null_start_only} ({(null_start_only / len(df_cabi)) * 100:.2f}%) of rides have only no start station.")

null_end_only = df_cabi[df_cabi["start_station_name"].notnull() & df_cabi["end_station_name"].isnull()].shape[0]
print(f"{null_end_only} ({(null_end_only / len(df_cabi)) * 100:.2f}%) of rides have only no end station.")

null_both = df_cabi[df_cabi["start_station_name"].isnull() & df_cabi["end_station_name"].isnull()].shape[0]
print(f"{null_both} ({(null_both / len(df_cabi)) * 100:.2f}%) of rides have neither a start nor an end station.")

null_either = df_cabi[df_cabi["start_station_name"].isnull() | df_cabi["end_station_name"].isnull()].shape[0]
print(f"{null_either} ({(null_either / len(df_cabi)) * 100:.2f}%) of ridess lack a start or an end station.")


62017 (9.16%) of rides have only no start station.
67089 (9.91%) of rides have only no end station.
38428 (5.68%) of rides have neither a start nor an end station.
167534 (24.75%) of ridess lack a start or an end station.


# Feature Additions

We seek to add the following features to the original data:

* `duration` - Duration of the ride in minutes

* `estimated_cost` - Cost of the trip, estimated using standard rates for casual or member riders

* `day_type` - Whether the trip was taken on a `weekday` or `weekend`

In [12]:
def estimate_cost(duration, rider_type, rideable, end_station):
    """
    Calculator for estimated cost of a ride based on duration, rider type, and rideable type.

    Args:
        duration (float): Duration of the ride in minutes.
        rider_type (str): Description of the rider type. Valid values are "member" or "casual".
        rideable (str): Description of the rideable type. Valid values are "classic_bike" and "electric_bike".
    
    For `casual` riders, bike unlocks are $1. 
        * Classic bikes are $0.15/min. 
        * Electric bikes are $0.35/min.
        * Parking an electric bike outside of a capital bikeshare station costs $3.

    For `member` riders, bike unlocks are free. 
        * Classic bikes are free for 45 minutes, then $0.05/min. 
        * Electric bikes are $0.15/min.
        * Parking an electric bike outside of a capital bikeshare station costs $3.
    """
    
    # duration is rounded down to the nearest minute
    duration = int(duration)
    
    # calculate cost based on rider type and rideable type
    if rider_type == "casual":
        if rideable == "classic_bike":
            return 1 + (0.15 * duration)
        
        elif rideable == "electric_bike":
            if end_station == None:
                return 1 + (0.35 * duration) + 3
            else:
                return 1 + (0.35 * duration)
            
        else:
            raise ValueError("Invalid rideable type. Valid values are 'classic_bike' or 'electric_bike'.")
    
    elif rider_type == "member":
        if rideable == "classic_bike":
            if duration <= 45:
                return 0
            else:
                return 0.05 * (duration - 45)
            
        elif rideable == "electric_bike":
            if end_station is None:
                return (0.15 * duration) + 3
            else:
                return 0.15 * duration
        else:
            raise ValueError("Invalid rideable type. Valid values are 'classic_bike' or 'electric_bike'.")
    
    else:
        raise ValueError("Invalid rider type. Valid values are 'member' or 'casual'.")

In [13]:
# add duration in minutes column
df_cabi["duration"] = (df_cabi["ended_at"] - df_cabi["started_at"]).dt.total_seconds() / 60

# add estimated cost column
df_cabi["estimated_cost"] = df_cabi.apply(
    lambda row: estimate_cost(row["duration"],
                              row["member_casual"],
                              row["rideable_type"],
                              row["end_station_name"] if pd.notnull(row["end_station_name"]) else None),
    axis=1
)

# add day type column
df_cabi["day_type"] = df_cabi["started_at"].apply(lambda x: "weekday" if x.weekday() < 5 else "weekend")

In [14]:
df_cabi.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration,estimated_cost,day_type
0,3FA776BA68AAC5F2,classic_bike,2026-09-13 13:29:42.674,2026-09-13 13:40:33.637,17th St & Potomac Ave SE / Congressional Cemet...,31727,17th St & Potomac Ave SE / Congressional Cemet...,31727,38.882458,-76.980715,38.882458,-76.980715,member,10.849383,0.00,weekend
1,0DDE689949EEE2AA,classic_bike,2026-09-13 16:02:48.322,2026-09-13 16:16:16.344,17th St & Potomac Ave SE / Congressional Cemet...,31727,17th St & Potomac Ave SE / Congressional Cemet...,31727,38.882458,-76.980715,38.882458,-76.980715,member,13.467033,0.00,weekend
2,4B620586F5D84EC8,electric_bike,2026-09-12 11:02:15.677,2026-09-12 13:26:22.000,Fenton St & New York Ave,32001,Fenton St & New York Ave,32001,38.985404,-77.023082,38.985404,-77.023082,casual,144.105383,51.40,weekend
3,CB1C2CADBB6C64AD,electric_bike,2026-09-01 20:49:14.502,2026-09-01 21:05:21.948,American University East Campus,31320,American University East Campus,31320,38.936298,-77.087128,38.936298,-77.087128,member,16.124100,2.40,weekday
4,E3F0AA83CFB29B45,electric_bike,2026-09-18 23:17:01.401,2026-09-18 23:22:47.884,MLK & Marion Barry Ave SE,31802,16th & Bangor St SE,31829,38.867373,-76.988039,38.860440,-76.983585,member,5.774717,0.75,weekday


In [15]:
df_cabi.shape

(676808, 16)

# Saving Preprocessed Data

Cleaned data is stored as `202609-cbs-tripdata-preprocessed.csv`.

In [16]:
# write to a csv file
df_cabi.to_csv("202609-cbs-tripdata-preprocessed.csv", index=False)